# SLC46A1 engraftment, step 6: the rescue search run in the construct background

Every rescuer in hand was selected against wild-type chicken, and step 5 showed
that three of the five do not hold once all twenty primary humanising
substitutions are present. The sequence those rescuers were chosen against no
longer exists in the finished surrogate.

Part A repeats the step 2 scan, every single substitution at every position,
against the construct background instead: all twenty primaries written in, with
only the target under test still carrying its chicken residue. The targets are
the ones step 5 reports below the threshold in that background, read from
data/construct_report.json rather than chosen here. Each substitution carries
its step 2 value beside the new one, so the effect of the background is measured
per substitution and not inferred.

Part B takes the best admissible rescuer per target, writes all of them into one
sequence together with the primaries, and rescores every primary with the others
in place, which is the step 5 measurement repeated on the new set. Two
admissibility policies are reported, one allowing any position and one skipping
the positions the pharmacophore set protects.

It stops on fatal conditions instead of warning, checkpoints to Drive after
every batch, and resumes from the checkpoint if the runtime is lost. The scan is
about two and a half hours on a T4. Run every cell in order.


In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step6')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/construct_report.json',
            'data/pharmacophore_position_set.tsv',
            'data/primary_substitution_risk.tsv',
            'data/parker_residues.json',
            'data/hfm_positions.tsv',
            'data/esm2_step2/rescue_scan.tsv',
            'structures/7BC6_to_human.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))
CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
EXPECT_SHA = '5d68e7c91dc3af3cd898a3f459617f3b22670f3dbdf9f798e8d740d8669a8aea'
if CHICKEN_SHA != EXPECT_SHA:
    raise SystemExit('FATAL the chicken sequence differs from the one steps 4 and 5 '
                     'scored: %s against %s' % (CHICKEN_SHA, EXPECT_SHA))
log('sequences ok, chicken_sha256 matches steps 4 and 5')

AA = 'ACDEFGHIKLMNPQRSTVWY'
THRESHOLD = 1e-3

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pcol = {k: i for i, k in enumerate(rows[0])}
PSET = {}
for r in rows[1:]:
    cp, hp = int(r[pcol['chicken_pos']]), int(r[pcol['human_pos']])
    if chicken[cp - 1] != r[pcol['chicken_aa']] or human[hp - 1] != r[pcol['human_aa']]:
        raise SystemExit('FATAL the position set disagrees with the sequences at chicken '
                         '%d' % cp)
    PSET[cp] = {'chicken_aa': r[pcol['chicken_aa']], 'human_aa': r[pcol['human_aa']],
                'human_pos': hp, 'differs': r[pcol['differs']] == '1',
                'parker': r[pcol['in_parker_named']] == '1',
                'role': r[pcol['role_or_note']]}
PRIM = sorted(p for p, v in PSET.items() if v['differs'])
if len(PRIM) != 20:
    raise SystemExit('FATAL %d primary substitutions, expected 20' % len(PRIM))

ck2hu = {int(k): v for k, v in
         json.load(open(os.path.join(PROJECT, 'structures', '7BC6_to_human.json')))['ck2hu'].items()}
hu2ck = {int(k): v for k, v in
         json.load(open(os.path.join(PROJECT, 'structures', '7BC6_to_human.json')))['hu2ck'].items()}
parker = json.load(open(os.path.join(PROJECT, 'data', 'parker_residues.json')))['residues']
PROTECTED = set()
for k, v in parker.items():
    cp, hp = int(k), v['human']
    if chicken[cp - 1] == human[hp - 1]:
        PROTECTED.add(cp)
unmapped = []
for r in [l.rstrip(chr(10)).split(chr(9))
          for l in open(os.path.join(PROJECT, 'data', 'hfm_positions.tsv'))][1:]:
    hp = int(r[0])
    cp = hu2ck.get(hp)
    if cp is None:
        unmapped.append(hp)
    else:
        PROTECTED.add(cp)
if unmapped:
    raise SystemExit('FATAL hereditary folate malabsorption positions without a chicken '
                     'partner: %s' % unmapped)
log('position set %d, primaries %d, protected positions %d'
    % (len(PSET), len(PRIM), len(PROTECTED)))


In [ ]:
con = json.load(open(os.path.join(PROJECT, 'data', 'construct_report.json')))
if abs(con['threshold'] - THRESHOLD) > 1e-12:
    raise SystemExit('FATAL step 5 used a threshold of %g, this notebook uses %g'
                     % (con['threshold'], THRESHOLD))
TARGETS = sorted(int(t) for t in con['part_b']['below_threshold']['primaries'])
if not TARGETS:
    raise SystemExit('FATAL step 5 reports no primary below the threshold in the '
                     'primaries background, so there is nothing to search for')
bad = [t for t in TARGETS if t not in PRIM]
if bad:
    raise SystemExit('FATAL step 5 names %s as below threshold, but they are not primary '
                     'substitutions' % bad)

PROF = {x['chicken_pos']: x for x in con['part_b']['profile']}
rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
rcol = {k: i for i, k in enumerate(rows[0])}
WT_BASE = {}
for r in rows[1:]:
    WT_BASE[int(r[rcol['chicken_pos']])] = float(r[rcol['p_human_wt_background']])

log('targets, read from the step 5 report: %s' % TARGETS)
log('  target  human  wild-type background  primaries background')
for t in TARGETS:
    log('  %6d %6s %21.6e %21.6e'
        % (t, PSET[t]['human_aa'], WT_BASE[t], PROF[t]['primaries']))


def background(leave_out):
    s = list(chicken)
    for p in PRIM:
        if p == leave_out:
            continue
        s[p - 1] = PSET[p]['human_aa']
    return ''.join(s)


variants = []
for t in TARGETS:
    bg = background(t)
    for j in range(1, len(chicken) + 1):
        if j == t:
            continue
        for y in AA:
            if y == bg[j - 1]:
                continue
            variants.append((t, j, y))
log('variants to score: %d over %d targets' % (len(variants), len(TARGETS)))


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v6-construct-rescue'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()
AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')
offset = 1

BG_IDS = {}
for t in TARGETS:
    ids = tok(background(t), return_tensors='pt')['input_ids'][0]
    if int(ids[t - 1 + offset]) != AAID[chicken[t - 1]]:
        raise SystemExit('FATAL the background for target %d does not carry the chicken '
                         'residue there' % t)
    for p in PRIM:
        if p == t:
            continue
        if int(ids[p - 1 + offset]) != AAID[PSET[p]['human_aa']]:
            raise SystemExit('FATAL the background for target %d does not carry the human '
                             'residue at primary %d' % (t, p))
    BG_IDS[t] = ids
log('backgrounds built and checked for %d targets' % len(BG_IDS))

CK = os.path.join(OUT, 'scan_partial.tsv')
HEADER = chr(9).join(['schema_' + SCHEMA, 'target', 'pos', 'aa', 'p_target'])
done = set()
if os.path.exists(CK) and os.path.getsize(CK) > 0:
    first = open(CK).readline().rstrip(chr(10))
    if first != HEADER:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; delete '
                         'it and rerun' % CK)
    for line in open(CK):
        p = line.rstrip(chr(10)).split(chr(9))
        if p[0] == 'row':
            done.add((int(p[1]), int(p[2]), p[3]))
    log('resuming, already scored %d' % len(done))

todo = [v for v in variants if v not in done]
log('scan: %d to score of %d' % (len(todo), len(variants)))
t0 = time.time()
with open(CK, 'a') as fh:
    if not done:
        fh.write(HEADER + chr(10))
        fh.flush()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        rowsb, idxs, toks = [], [], []
        batch = torch.stack([BG_IDS[t].clone() for (t, j, y) in chunk])
        for k, (t, j, y) in enumerate(chunk):
            batch[k, j - 1 + offset] = AAID[y]
            idxs.append(t - 1 + offset)
            toks.append(AAID[PSET[t]['human_aa']])
        with torch.no_grad():
            logits = model(batch.cuda()).logits
        probs = torch.softmax(logits.float(), dim=-1)
        for k, (t, j, y) in enumerate(chunk):
            fh.write(chr(9).join(['row', str(t), str(j), y,
                                  '%.6e' % float(probs[k, idxs[k], toks[k]])]) + chr(10))
        fh.flush()
        if (b // BATCH) % 50 == 0:
            el = (time.time() - t0) / 60
            rate = (b + len(chunk)) / max(el, 1e-9)
            log('  %d of %d, %.1f min elapsed, about %.0f min left'
                % (b + len(chunk), len(todo), el,
                   (len(todo) - b - len(chunk)) / max(rate, 1e-9)))
log('scan complete, %.1f min' % ((time.time() - t0) / 60))


In [ ]:
S = {}
for line in open(CK):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        S[(int(p[1]), int(p[2]), p[3])] = float(p[4])
if len(S) != len(variants):
    raise SystemExit('FATAL the checkpoint holds %d rows, %d expected'
                     % (len(S), len(variants)))

PREV = {}
scan = open(os.path.join(PROJECT, 'data', 'esm2_step2', 'rescue_scan.tsv'))
head = scan.readline().rstrip(chr(10)).split(chr(9))
scol = {k: i for i, k in enumerate(head)}
for line in scan:
    r = line.rstrip(chr(10)).split(chr(9))
    PREV[(int(r[scol['target_pos']]), int(r[scol['rescuer_pos']]),
          r[scol['rescuer_to']])] = float(r[scol['p_target']])
if not PREV:
    raise SystemExit('FATAL the step 2 scan produced no rows')

cols = ['target', 'pos', 'from_construct', 'from_chicken', 'to', 'separation',
        'p_construct', 'p_wt_step2', 'fold_construct_over_wt',
        'crosses_construct', 'crosses_wt', 'is_primary', 'protected']
out = []
no_prev = 0
for (t, j, y), v in S.items():
    bg = background(t)
    pw = PREV.get((t, j, y))
    if pw is None:
        no_prev += 1
    out.append({'target': t, 'pos': j, 'from_construct': bg[j - 1],
                'from_chicken': chicken[j - 1], 'to': y, 'separation': abs(j - t),
                'p_construct': v, 'p_wt_step2': pw,
                'fold_construct_over_wt': (v / pw) if pw else None,
                'crosses_construct': v >= THRESHOLD,
                'crosses_wt': (pw >= THRESHOLD) if pw else None,
                'is_primary': j in PRIM, 'protected': j in PROTECTED})
out.sort(key=lambda r: (r['target'], -r['p_construct']))
with open(os.path.join(OUT, 'construct_rescue_scan.tsv'), 'w') as fh:
    fh.write(chr(9).join(cols) + chr(10))
    for r in out:
        fh.write(chr(9).join(
            ('%.6e' % r[c]) if isinstance(r[c], float)
            else ('' if r[c] is None else str(r[c])) for c in cols) + chr(10))
log('wrote construct_rescue_scan.tsv, %d rows, %d without a step 2 counterpart'
    % (len(out), no_prev))

log('')
log('per target, scored against the construct background')
log('  target  baseline here  crossing  best substitution        p        step 2 value  '
    'fold')
PERT = {}
for t in TARGETS:
    here = [r for r in out if r['target'] == t]
    cross = [r for r in here if r['crosses_construct']]
    best = max(here, key=lambda r: r['p_construct'])
    PERT[t] = {'baseline_construct': PROF[t]['primaries'],
               'baseline_wt': WT_BASE[t], 'n_scored': len(here),
               'n_crossing': len(cross), 'best': best,
               'crossing': cross[:50]}
    log('  %6d %14.6e %9d  %s%d%s %18.6e %14s %6s'
        % (t, PROF[t]['primaries'], len(cross), best['from_construct'], best['pos'],
           best['to'], best['p_construct'],
           ('%.6e' % best['p_wt_step2']) if best['p_wt_step2'] else '-',
           ('%.2f' % best['fold_construct_over_wt'])
           if best['fold_construct_over_wt'] else '-'))

both = [r for r in out if r['p_wt_step2']]
gained = [r for r in both if r['crosses_construct'] and not r['crosses_wt']]
lost = [r for r in both if r['crosses_wt'] and not r['crosses_construct']]
log('')
log('the background changes which substitutions cross: %d cross here and not in step 2, '
    '%d cross in step 2 and not here, of %d comparable rows'
    % (len(gained), len(lost), len(both)))
folds = sorted(r['fold_construct_over_wt'] for r in both)
log('fold of the construct value over the step 2 value: median %.2f, 5th and 95th '
    'percentile %.2f and %.2f'
    % (folds[len(folds) // 2], folds[int(0.05 * len(folds))],
       folds[int(0.95 * len(folds))]))


In [ ]:
POLICIES = ['published', 'annotated']


def admissible(policy, j, t):
    if j == t:
        return False
    if policy == 'published':
        return True
    if policy == 'annotated':
        return j not in PROTECTED and j not in PRIM
    raise SystemExit('FATAL unknown policy ' + policy)


CHOSEN = {}
for policy in POLICIES:
    picks = []
    for t in TARGETS:
        here = [r for r in out if r['target'] == t and admissible(policy, r['pos'], t)]
        cross = [r for r in here if r['crosses_construct']]
        if not cross:
            picks.append({'target': t, 'found': False})
            continue
        b = max(cross, key=lambda r: r['p_construct'])
        picks.append({'target': t, 'found': True, 'pos': b['pos'], 'to': b['to'],
                      'from_construct': b['from_construct'],
                      'from_chicken': b['from_chicken'],
                      'p_construct': b['p_construct'], 'separation': b['separation'],
                      'is_primary': b['is_primary'], 'protected': b['protected']})
    CHOSEN[policy] = picks
    log('')
    log('policy %s' % policy)
    log('  target  rescuer            separation  p in the construct  primary  protected')
    for p in picks:
        if not p['found']:
            log('  %6d  %-18s %10s %19s %8s %10s'
                % (p['target'], 'none crossing', '-', '-', '-', '-'))
            continue
        log('  %6d  %s%d%s %28s %19.6e %8s %10s'
            % (p['target'], p['from_construct'], p['pos'], p['to'], p['separation'],
               p['p_construct'], p['is_primary'], p['protected']))
    clash = {}
    for p in picks:
        if p['found']:
            clash.setdefault(p['pos'], set()).add(p['to'])
    multi = {k: v for k, v in clash.items() if len(v) > 1}
    if multi:
        log('  two rescuers want different residues at the same position: %s, so the '
            'weaker one is dropped' % multi)
        keep, seen = [], {}
        for p in sorted([q for q in picks if q['found']],
                        key=lambda q: -q['p_construct']):
            if p['pos'] in seen:
                continue
            seen[p['pos']] = True
            keep.append(p)
        CHOSEN[policy] = keep + [q for q in picks if not q['found']]


def build(policy, leave_out=None):
    s = list(chicken)
    for p in PRIM:
        if p == leave_out:
            continue
        s[p - 1] = PSET[p]['human_aa']
    for r in CHOSEN[policy]:
        if not r['found'] or r['pos'] == leave_out:
            continue
        s[r['pos'] - 1] = r['to']
    return ''.join(s)


for policy in POLICIES:
    s = build(policy)
    log('construct under %s differs from chicken at %d positions'
        % (policy, sum(1 for a, b in zip(s, chicken) if a != b)))

jobs = [(policy, t) for policy in POLICIES for t in PRIM]
CK7 = os.path.join(OUT, 'profile_partial.tsv')
H7 = chr(9).join(['schema_' + SCHEMA, 'policy', 'chicken_pos', 'p_human'])
done7 = set()
if os.path.exists(CK7) and os.path.getsize(CK7) > 0:
    if open(CK7).readline().rstrip(chr(10)) != H7:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; delete '
                         'it and rerun' % CK7)
    for line in open(CK7):
        p = line.rstrip(chr(10)).split(chr(9))
        if p[0] == 'row':
            done7.add((p[1], int(p[2])))
todo7 = [j for j in jobs if j not in done7]
log('profile scoring: %d of %d to do' % (len(todo7), len(jobs)))
with open(CK7, 'a') as fh:
    if not done7:
        fh.write(H7 + chr(10))
        fh.flush()
    for b in range(0, len(todo7), BATCH):
        chunk = todo7[b:b + BATCH]
        seqs = [build(policy, t) for (policy, t) in chunk]
        enc = tok(seqs, return_tensors='pt')['input_ids']
        if enc.shape[1] != len(chicken) + 2:
            raise SystemExit('FATAL a constructed sequence tokenizes to %d tokens, '
                             'expected %d' % (enc.shape[1], len(chicken) + 2))
        with torch.no_grad():
            logits = model(enc.cuda()).logits
        probs = torch.softmax(logits.float(), dim=-1)
        for k, (policy, t) in enumerate(chunk):
            idx = t - 1 + offset
            if int(enc[k, idx]) != AAID[chicken[t - 1]]:
                raise SystemExit('FATAL the left-out position %d does not carry the '
                                 'chicken residue under %s' % (t, policy))
            fh.write(chr(9).join(['row', policy, str(t),
                                  '%.6e' % float(probs[k, idx,
                                                       AAID[PSET[t]['human_aa']]])])
                     + chr(10))
        fh.flush()
log('profile scoring complete')


In [ ]:
PR = {}
for line in open(CK7):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        PR[(p[1], int(p[2]))] = float(p[3])
if len(PR) != len(jobs):
    raise SystemExit('FATAL the profile checkpoint holds %d rows, %d expected'
                     % (len(PR), len(jobs)))

pcols = ['chicken_pos', 'human_aa', 'p_wt_step1', 'p_primaries_step5',
         'p_step5_final'] + ['p_' + x for x in POLICIES]
prows = []
for t in PRIM:
    rec = {'chicken_pos': t, 'human_aa': PSET[t]['human_aa'],
           'p_wt_step1': WT_BASE[t],
           'p_primaries_step5': PROF[t]['primaries'],
           'p_step5_final': PROF[t]['primaries_rescuers_primary_wins']}
    for policy in POLICIES:
        rec['p_' + policy] = PR[(policy, t)]
    prows.append(rec)
with open(os.path.join(OUT, 'construct_rescue_profile.tsv'), 'w') as fh:
    fh.write(chr(9).join(pcols) + chr(10))
    for r in prows:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in pcols) + chr(10))

log('')
log('every primary with the others in place, under each policy')
log('chicken  human   wild type  primaries   step 5 final  %s'
    % '  '.join('%-12s' % p for p in POLICIES))
for r in prows:
    log('%7d %6s %11.3e %11.3e %14.3e  %s'
        % (r['chicken_pos'], r['human_aa'], r['p_wt_step1'], r['p_primaries_step5'],
           r['p_step5_final'],
           '  '.join('%-12.3e' % r['p_' + p] for p in POLICIES)))

log('')
for key, label in ([('p_step5_final', 'step 5 final')]
                   + [('p_' + p, p) for p in POLICIES]):
    below = [r['chicken_pos'] for r in prows if r[key] < THRESHOLD]
    log('%-28s below %.0e at %d of %d primaries%s'
        % (label, THRESHOLD, len(below), len(prows), (': %s' % below) if below else ''))

summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'threshold': THRESHOLD, 'targets': TARGETS, 'policies': POLICIES,
           'n_variants_scored': len(S),
           'background_effect': {'n_comparable': len(both),
                                 'crossing_here_not_step2': len(gained),
                                 'crossing_step2_not_here': len(lost),
                                 'fold_median': folds[len(folds) // 2],
                                 'fold_p05': folds[int(0.05 * len(folds))],
                                 'fold_p95': folds[int(0.95 * len(folds))]},
           'per_target': {str(t): {'baseline_construct': PERT[t]['baseline_construct'],
                                   'baseline_wt': PERT[t]['baseline_wt'],
                                   'n_scored': PERT[t]['n_scored'],
                                   'n_crossing': PERT[t]['n_crossing'],
                                   'best': PERT[t]['best'],
                                   'crossing_top': PERT[t]['crossing']}
                          for t in TARGETS},
           'chosen': CHOSEN,
           'profile': prows,
           'below_threshold': {k: [r['chicken_pos'] for r in prows if r[k] < THRESHOLD]
                               for k in ['p_step5_final'] + ['p_' + p for p in POLICIES]}}
with open(os.path.join(OUT, 'construct_rescue_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
log('')
log('wrote construct_rescue_scan.tsv, construct_rescue_profile.tsv and '
    'construct_rescue_summary.json')
log('copy esm2_step6/ from Drive into data/ and run codes/19_construct_rescue_read.py')
